# Taller Despliegue Flask API, PARTE I: Local en CSV

Para esta primera parte vamos a desplegar un modelo de machine learning en una API **para su consumo en local** (como hemos visto en el taller anterior para la base de datos de novelas de ciencia ficción). 

Entrenaremos un modelo, lo guardaremos entrenado, y desarrollaremos una API que permita consumir dicho modelo desde cualquier otra tecnología, pero primero en un servicio local.

Para que tengas más contexto... **Se presenta el siguiente caso de uso**

Una empresa distribuidora de muebles del ámbito nacional pretende utilizar un modelo desarrollado por el departamento de analítica, con el que consiguen una predicción de las ventas a partir de los gastos en marketing de anuncios en televisión, radio y periódicos. Quieren incorporar estos datos dentro de su página web interna, donde comparten todo tipo de información relativa a resultados de la empresa, ventas, adquisiciones, etc... La web está desarrollada en AngularJS, mientras que el modelo se desarrolló en Python, por lo que precisamos de una interfaz de comunicación entre ambos sistemas.

El equipo de desarrollo necesita que implementes un microservicio para que ellos puedan consumir el modelo desde la propia web. El microservicio tiene que cumplir las siguientes características:
1. Ofrezca la predicción de ventas a partir de todos los valores de gastos en publicidad.
2. Podamos actualizar la base de datos con nuevos registros, una vez conozcamos los valores de venta reales.
3. Posibilidad de reentrenar el modelo con los nuevos registros.

**¿Qué es necesario implementar, y se deja como ejercicio opcional?**  

1. Por simplicidad del ejercicio, la base de datos con la que se trabaja es un CSV ("Advertising.csv")

2. El modelo. En el momento de reentrenamiento, simplemente se entrena la regresión lineal con los últimos datos. No es necesario comprobar si los resultados son mejores o peores, tenemos missings u outliers...

3. Queremos implementar 3 endpoints: El del mensaje local para el acceso al /, un endpoint que dados los valores de inversión en TV, Radio y Periódicos nos diga las ventas esperadas y otro endpoint en el que el servidor de la API compruebe si tiene datos nuevos y reentrene el modelo y vuelva a calcular sus métricas. (Suponemos que el fichero)

**NOTA**: Cuentas con un script de Python (*model.py*) con el código de entrenamiento del modelo ya hecho, puesto que el desarrollo de un modelo de machine learning no es el objetivo del ejercicio, sino el diseño de una API con Flask

### Procedimiento

**#1. Creación del modelo**

Carga el script `model.py`, revisalo y ejecútalo para crear nuestro modelo.

El script `model.py` hace lo siguiente:

1. Lee el CSV `Advertising.csv` con las columnas `TV`, `radio`, `newspaper` (inversión en publicidad) y `sales` (ventas).
2. Divide los datos en train (80%) y test (20%).
3. Entrena un modelo **Lasso** con `alpha=6000`. Lasso es una regresión lineal con regularización: penaliza coeficientes grandes y puede llevar a cero los de variables poco útiles.
4. Evalúa el modelo con cross-validation de 4 folds sobre train, y también sobre el conjunto test. Las métricas usadas son MSE, RMSE y MAPE.
5. Una vez evaluado, **reentrena con el 100% de los datos** para aprovechar toda la información disponible antes de guardar.
6. Serializa el modelo entrenado en `ad_model.pkl` con `pickle.dump()`. Serializar significa convertir el objeto Python (con todos sus coeficientes) a bytes y guardarlo en disco, para poder cargarlo después sin tener que reentrenar.

Resultado al ejecutarlo:
```
Train Mean Sales 14100.0
MSE Cross:  3126310.47
RMSE Cross:  1752.07
MAPE Cross:  0.136
**********
MSE Test:  3213225.62
RMSE Test:  1792.55
MAPE Test:  0.143
```
Con una media de ventas de 14.100 unidades y un RMSE de ~1.750, el modelo se equivoca de media en ±1.750 unidades (~12% del valor medio). El archivo `ad_model.pkl` queda guardado en la carpeta del proyecto.

**#2. Creación del script para dar servicio a la API (modo local)**

Completa el script "app_model.py" añadiendo el routing a las siguientes funciones (revísalas antes) (en los comentarios de cada función se describe el endpoint)

```python
def hello(): # Ligado al endopoint "/" o sea el home, con el método GET
    return "Bienvenido a mi API del modelo advertising"
```

Una **API** es una interfaz que permite que dos sistemas distintos se comuniquen: en este caso, la web en AngularJS y el modelo en Python. **Flask** es el microframework que nos permite construir esa interfaz con muy poco código.

El **routing** es el mecanismo que conecta una URL con una función Python. En Flask se hace con el decorador `@app.route()`, que funciona como un letrero: "cuando llegue una petición a esta URL con este método HTTP, ejecuta esta función".

```python
@app.route("/", methods=["GET"])
def hello():
    return "Bienvenido a mi API del modelo advertising"
```

Este es el endpoint raíz. Sirve para comprobar que el servidor está vivo. Al hacer `GET http://127.0.0.1:5000/` desde el navegador, Flask ejecuta `hello()` y devuelve el mensaje.

```python
def predict(): # Ligado al endpoint '/api/v1/predict', con el método GET

    model = pickle.load(open('ad_model.pkl','rb'))
    tv = request.args.get('tv', None)
    radio = request.args.get('radio', None)
    newspaper = request.args.get('newspaper', None)

    print(tv,radio,newspaper)
    print(type(tv))

    if tv is None or radio is None or newspaper is None:
        return "Args empty, the data are not enough to predict"
    else:
        prediction = model.predict([[float(tv),float(radio),float(newspaper)]])
    
    return jsonify({'predictions': prediction[0]})
```

Línea por línea:

- `pickle.load(open('ad_model.pkl', 'rb'))`: carga el modelo desde disco en modo lectura binaria (`'rb'`). Se carga en cada petición para que si el modelo fue reentrenado (endpoint `/retrain`), la siguiente predicción ya use la versión nueva.

- `request.args.get('tv', None)`: lee el parámetro `tv` del **query string** de la URL. Los parámetros de query string son los que van después del `?` en la URL, separados por `&`. Por ejemplo: `http://127.0.0.1:5000/api/v1/predict?tv=150&radio=30&newspaper=20`. El segundo argumento `None` indica el valor por defecto si el parámetro no existe.

- `float(tv)`: los valores del query string llegan siempre como **strings** aunque sean números (`'150'`, no `150`). Hay que convertirlos a float antes de pasárselos al modelo.

- `pd.DataFrame([[...]], columns=[...])`: construimos un DataFrame con los nombres de columna exactos con los que se entrenó el modelo (`TV`, `radio`, `newspaper`). Si pasáramos una lista plana `[[tv, radio, newspaper]]`, sklearn lanzaría un warning porque el modelo fue entrenado con un DataFrame con nombres de columna y espera recibir lo mismo. El resultado numérico sería correcto, pero es mala práctica ignorar ese aviso.

- `jsonify({'predictions': prediction[0]})`: convierte el resultado a JSON, que es el formato estándar para que las APIs devuelvan datos estructurados. `prediction[0]` extrae el valor escalar del array numpy.

Para la función anterior, ¿Cómo crees que espera los argumentos la API? (¿como parámetros en el cuerpo de la petición, como querystring,...?)

Los espera como **query string parameters** en la URL. La evidencia está en `request.args.get()`: `request.args` es el objeto de Flask específico para leer parámetros de query string. Si los argumentos vinieran en el cuerpo de la petición (como haría un POST con JSON), usaríamos `request.get_json()` en su lugar.

La llamada correcta es:
```
GET http://127.0.0.1:5000/api/v1/predict?tv=150&radio=30&newspaper=20
```

```python
def retrain(): # Rutarlo al endpoint '/api/v1/retrain/', metodo GET
    if os.path.exists("data/Advertising_new.csv"):
        data = pd.read_csv('data/Advertising_new.csv')

        X_train, X_test, y_train, y_test = train_test_split(data.drop(columns=['sales']),
                                                        data['sales'],
                                                        test_size = 0.20,
                                                        random_state=42)

        model = Lasso(alpha=6000)
        model.fit(X_train, y_train)
        rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
        mape = mean_absolute_percentage_error(y_test, model.predict(X_test))
        model.fit(data.drop(columns=['sales']), data['sales'])
        pickle.dump(model, open('ad_model.pkl', 'wb'))

        return f"Model retrained. New evaluation metric RMSE: {str(rmse)}, MAPE: {str(mape)}"
    else:
        return f"<h2>New data for retrain NOT FOUND. Nothing done!</h2>"

```

- `os.path.exists("data/Advertising_new.csv")`: comprueba si existe el archivo con datos nuevos antes de hacer nada. Si no existe, devuelve el mensaje de aviso sin tocar el modelo actual.

- El flujo de reentrenamiento es idéntico al de `model.py`: split → entrena → evalúa sobre test → reentrena con el 100% → sobreescribe `ad_model.pkl`. A partir de ese momento, cualquier nueva petición a `/predict` usará el modelo actualizado.

- `pickle.dump(model, open('ad_model.pkl', 'wb'))`: sobreescribe el archivo existente con el modelo reentrenado. El modo `'wb'` (write binary) crea el archivo si no existe o lo sobreescribe si ya existe.

Al ejecutarlo con `Advertising_new.csv` (275 filas, 75 más que el original):
```
Model retrained. New evaluation metric RMSE: 2126.69, MAPE: 0.138
```

**#3. Hora de ejecutar y probar varias predicciones y el endpoint de retrain**

Para ejecutar la API localmente:

1. Primero ejecutar `model.py` para generar `ad_model.pkl`:
```bash
python model.py
```

2. Levantar el servidor Flask:
```bash
python app_model.py
```

3. Probar los endpoints desde el navegador o con `requests`:

In [ ]:
import requests

# probamos el endpoint home
r = requests.get('http://127.0.0.1:5000/')
print(r.text)

# probamos varias predicciones con distintas inversiones publicitarias
casos = [
    {'tv': 150, 'radio': 30, 'newspaper': 20},
    {'tv': 230, 'radio': 37, 'newspaper': 69},
    {'tv': 44,  'radio': 39, 'newspaper': 45},
    {'tv': 300, 'radio': 5,  'newspaper': 5},
]

for params in casos:
    r = requests.get('http://127.0.0.1:5000/api/v1/predict', params=params)
    print(f"tv={params['tv']}, radio={params['radio']}, newspaper={params['newspaper']}  →  {r.json()}")

# probamos sin parámetros — debería devolver el mensaje de error
r = requests.get('http://127.0.0.1:5000/api/v1/predict')
print(r.text)

# probamos el reentrenamiento
r = requests.get('http://127.0.0.1:5000/api/v1/retrain/')
print(r.text)